# SR1.5 feasibility scorer (simple)

Self-contained notebook. Reads only the canonical files and regenerates the
derived ones (`sr15_feasibility_dim_aggregate.csv`, `sr15_cell_bridge.csv`,
`sr15_to_city_mapping.csv`, `ranked_actions.csv`).

**Rule:** city-data adjustment only on cells where SR1.5 code is A or C.

In [2]:
import pandas as pd

BASE = './data'
actions     = pd.read_csv(f'{BASE}/actions_to_sr15_mapping.csv')
per_cell    = pd.read_csv(f'{BASE}/sr15_feasibility_per_cell.csv')
ind_bridge  = pd.read_csv(f'{BASE}/sr15_indicator_to_city_indicator.csv')
city_df     = pd.read_csv(f'{BASE}/sample_chile_indicators.csv')

QUINT = {'very low':0.0, 'low':0.25, 'medium':0.5, 'high':0.75, 'very high':1.0}

OPTION_FAMILY = {
    'Wind (on-shore & off-shore)':'energy_supply','Solar PV':'energy_supply','Bioenergy':'energy_supply',
    'Electricity storage':'energy_supply','Power sector CCS':'energy_supply','Nuclear energy':'energy_supply',
    'Smart grids':'energy_supply','BECCS':'energy_supply','DACCS':'energy_supply',
    'Reduced food wastage':'waste','Dietary shifts':'waste',
    'Sustainable intensification':'nbs','Afforestation & reforestation':'nbs',
    'Soil carbon sequestration & biochar':'nbs','Enhanced weathering':'nbs',
    'Land-use & urban planning':'cross_cutting',
    'Electric cars and buses':'transport','Sharing schemes':'transport','Public transport':'transport',
    'Non-motorised transport':'transport','Aviation & shipping':'transport',
    'Efficient appliances':'buildings','Low/zero-energy buildings':'buildings',
    'Energy efficiency':'industrial','Bio-based & circularity':'industrial',
    'Electrification & hydrogen':'industrial','Industrial CCS':'industrial',
}
DATASOURCE = {
    'median_household_income':'cl-casen 2022','poverty_rate':'cl-casen 2022',
    'unemployment_rate':'cl-casen 2022','public_transport_share':'cl-casen 2022',
    'electricity_access_rate':'cl-ine-censo 2024','home_ownership':'cl-ine-censo 2024',
    'renter_share':'cl-ine-censo 2024','industry_construction_employment':'cl-ine-censo 2024',
    'employment_in_transport_and_logistics':'cl-ine-censo 2024',
    'mean_years_schooling':'cl-ine-censo 2024','literacy_rate':'cl-ine-censo 2024',
    'disability_prevalence':'cl-ine-censo 2024',
    'indigenous_identification_rate':'cl-ine-censo 2024',
    'fixed_internet_household_share':'cl-ine-censo 2024',
    'employment_agriculture_utilities':'cl-ine-censo 2024',
}
SECTION = {
    '4.SM.7':'Energy — renewables','4.SM.8':'Energy — storage/CCS/nuclear',
    '4.SM.9':'Land & ecosystem','4.SM.10':'Urban — planning, EV, sharing',
    '4.SM.11':'Urban — transit, active, aviation','4.SM.12':'Urban — grid, appliances, buildings',
    '4.SM.13':'Industrial','4.SM.14':'CDR — BECCS, DACCS','4.SM.15':'CDR — afforestation, soil, weathering',
}

## Regenerate derived files from canonical inputs

In [3]:
# 1) Dimension-level aggregate — SR1.5 Table 4.SM.5 formula
dim_rows = []
for (opt, dim), g in per_cell.groupby(['option','dimension']):
    c = g['code'].value_counts()
    nA, nB, nC = c.get('A',0), c.get('B',0), c.get('C',0)
    nNA, nNE, nLE = c.get('NA',0), c.get('NE',0), c.get('LE',0)
    n_eff = len(g) - nNA
    n_NELE = nNE + nLE
    if n_eff == 0:
        avg, band = None, 'NA'
    elif n_NELE > 0.5 * n_eff:
        avg, band = None, 'insufficient_evidence'
    else:
        avg = (1*nA + 2*nB + 3*nC) / n_eff
        band = 'low' if avg<=1.5 else ('medium' if avg<=2.5 else 'high')
    dim_rows.append({'option':opt,'dimension':dim,'n_indicators':len(g),
                     'n_A':nA,'n_B':nB,'n_C':nC,'n_NA':nNA,'n_NE':nNE,'n_LE':nLE,
                     'avg_score': round(avg,3) if avg is not None else '','band':band})
dim_agg = pd.DataFrame(dim_rows)
# dim_agg.to_csv(f'{BASE}/sr15_feasibility_dim_aggregate.csv', index=False)

# 2) Cell-level bridge — expand A/C cells with applicable city bridges
real_bridges = ind_bridge[ind_bridge['city_indicator']!='(option-level constant)']
cb_rows = []
for _, cell in per_cell.iterrows():
    opt, ind, code, dim = cell['option'], cell['indicator'], cell['code'], cell['dimension']
    family = OPTION_FAMILY.get(opt, '?')
    if code not in ('A','C'):
        cb_rows.append({'option':opt,'dimension':dim,'indicator':ind,'sr15_code':code,
                        'option_family':family,'city_indicator':f'(SR1.5 code {code}: no directional evidence)',
                        'sign':0,'scope':''})
        continue
    apps = real_bridges[(real_bridges['sr15_indicator']==ind) & (real_bridges['scope'].isin(['all', family]))]
    if apps.empty:
        cb_rows.append({'option':opt,'dimension':dim,'indicator':ind,'sr15_code':code,
                        'option_family':family,'city_indicator':'(no scope-matching bridge)',
                        'sign':0,'scope':''})
    else:
        for _, b in apps.iterrows():
            cb_rows.append({'option':opt,'dimension':dim,'indicator':ind,'sr15_code':code,
                            'option_family':family,'city_indicator':b['city_indicator'],
                            'sign':int(b['sign']),'scope':b['scope']})
cell_bridge = pd.DataFrame(cb_rows)
# cell_bridge.to_csv(f'{BASE}/sr15_cell_bridge.csv', index=False)

# 3) to_city_mapping — slim consumer view with sm_section + datasource
sm_table_by_opt = per_cell.groupby('option')['table'].first().to_dict()
tcm_rows = []
for _, r in cell_bridge.iterrows():
    is_varying = 'yes' if r['sign'] in (1,-1) else 'no'
    ds = DATASOURCE.get(r['city_indicator'], '') if is_varying=='yes' else ''
    tcm_rows.append({'sm_section': SECTION.get(sm_table_by_opt.get(r['option'],''), ''),
                     'option':r['option'],'dimension':r['dimension'],'indicator':r['indicator'],
                     'is_city_varying':is_varying,'city_indicator':r['city_indicator'],
                     'city_indicator_datasource':ds,'sr15_code':r['sr15_code'],'sign':r['sign']})
to_city_mapping = pd.DataFrame(tcm_rows)
# to_city_mapping.to_csv(f'{BASE}/sr15_to_city_mapping.csv', index=False)

print(f'Wrote sr15_feasibility_dim_aggregate.csv ({len(dim_agg)} rows)')
print(f'Wrote sr15_cell_bridge.csv ({len(cell_bridge)} rows)')
print(f'Wrote sr15_to_city_mapping.csv ({len(to_city_mapping)} rows)')
n_active = sum(1 for r in cb_rows if r['sign'] in (1,-1))
print(f'  Active city-signal rows: {n_active}')

Wrote sr15_feasibility_dim_aggregate.csv (160 rows)
Wrote sr15_cell_bridge.csv (592 rows)
Wrote sr15_to_city_mapping.csv (592 rows)
  Active city-signal rows: 112


## Score an action for a city

In [4]:
def score_action(action_id, locode):
    a = actions[actions['action_id']==action_id].iloc[0]
    options = [o for o in a['sr15_options'].split('|') if not o.startswith('(')]
    if not options: return None, pd.DataFrame()
    cdf = city_df[city_df['locode']==locode]
    city_q = {r['attribute_type']: QUINT[r['attribute_category']] for _,r in cdf.iterrows()}
    ac = per_cell[(per_cell['option'].isin(options)) & (per_cell['code'].isin(['A','C']))]
    if ac.empty: return None, pd.DataFrame()
    rows = []
    for _, c in ac.iterrows():
        sr15 = +1 if c['code']=='C' else -1
        cb = cell_bridge[(cell_bridge['option']==c['option']) & (cell_bridge['indicator']==c['indicator'])
                         & (cell_bridge['sign'].isin([1,-1]))]
        contribs = []
        for _, b in cb.iterrows():
            ci = b['city_indicator']
            if ci in city_q:
                contribs.append(int(b['sign']) * (2*city_q[ci]-1))
        score = (sr15 + sum(contribs)/len(contribs))/2 if contribs else sr15
        rows.append({'option':c['option'],'dim':c['dimension'],'indicator':c['indicator'],
                     'code':c['code'],'cell_score_01':(score+1)/2})
    cells = pd.DataFrame(rows)
    return cells.groupby('dim')['cell_score_01'].mean().mean(), cells

## Walk through one action

In [5]:
score, cells = score_action('c40_0010', 'CL CNE')
print(f'Score: {score:.3f}')
cells

Score: 0.824


,option,dim,indicator,code,cell_score_01
0,Low/zero-energy buildings,economic,cost-effectiveness,C,0.500000
1,Low/zero-energy buildings,economic,distributional_effects,C,0.541667
2,Low/zero-energy buildings,economic,employment_productivity,C,0.812500
3,Low/zero-energy buildings,technological,technical_scalability,C,0.500000
4,Low/zero-energy buildings,institutional,political_acceptability,C,1.000000
5,Low/zero-energy buildings,environmental,toxic_waste,C,1.000000
6,Low/zero-energy buildings,environmental,water_use,C,1.000000
7,Low/zero-energy buildings,geophysical,physical_feasibility,C,1.000000


## Rank all actions for Colchane (full chain)

In [6]:
results = []
per_cell_rows = []
for aid in actions['action_id']:
    score, cells = score_action(aid, 'CL CNE')
    if score is None:
        continue
    a = actions[actions['action_id']==aid].iloc[0]
    cdf = city_df[city_df['locode']=='CL CNE']
    city_cat = {r['attribute_type']: r['attribute_category'] for _,r in cdf.iterrows()}
    city_q = {r['attribute_type']: QUINT[r['attribute_category']] for _,r in cdf.iterrows()}
    dim_scores = cells.groupby('dim')['cell_score_01'].mean().to_dict()
    results.append({
        'action_id':       aid,
        'action_name':     a['action_name'][:60],
        'primary_outcome': a['primary_outcome'],
        'primary_channel': a['primary_channel'],
        'sr15_options':    a['sr15_options'],
        'match_strength':  a['match_strength'],
        'n_ac_cells':      len(cells),
        'n_dims_scored':   len(dim_scores),
        'econ':            round(dim_scores.get('economic',       float('nan')), 2),
        'tech':            round(dim_scores.get('technological',  float('nan')), 2),
        'inst':            round(dim_scores.get('institutional',  float('nan')), 2),
        'soc':             round(dim_scores.get('socio_cultural', float('nan')), 2),
        'env':             round(dim_scores.get('environmental',  float('nan')), 2),
        'geo':             round(dim_scores.get('geophysical',    float('nan')), 2),
        'score':           round(score, 3),
    })
    # Per-indicator detail: one row per (action, cell, city_indicator).
    # Cells with no bridge get a single row with city_indicator='(no bridge)'.
    for _, c in cells.iterrows():
        cb = cell_bridge[(cell_bridge['option']==c['option']) & (cell_bridge['indicator']==c['indicator'])
                         & (cell_bridge['sign'].isin([1,-1]))]
        base_row = {
            'action_id':       aid,
            'action_name':     a['action_name'][:60],
            'sr15_option':     c['option'],
            'dimension':       c['dim'],
            'indicator':       c['indicator'],
            'sr15_code':       c['code'],
            'cell_score_01':   round(c['cell_score_01'], 3),
        }
        if cb.empty:
            per_cell_rows.append({**base_row,
                'city_indicator':            '(no bridge)',
                'sign':                      0,
                'city_quintile_category':    '',
                'city_capacity':             '',
                'indicator_contribution':    '',
            })
        else:
            for _, b in cb.iterrows():
                ci = b['city_indicator']
                if ci in city_q:
                    q = city_q[ci]
                    cat = city_cat[ci]
                    contrib = round(int(b['sign']) * (2*q - 1), 3)
                else:
                    q, cat, contrib = '', '(no city data)', ''
                per_cell_rows.append({**base_row,
                    'city_indicator':            ci,
                    'sign':                      int(b['sign']),
                    'city_quintile_category':    cat,
                    'city_capacity':             q,
                    'indicator_contribution':    contrib,
                })

ranked = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
ranked.to_csv(f'{BASE}/ranked_actions.csv', index=False)

per_cell_df = pd.DataFrame(per_cell_rows)
score_lookup = dict(zip(ranked['action_id'], ranked['score']))
per_cell_df['_score'] = per_cell_df['action_id'].map(score_lookup)
per_cell_df = per_cell_df.sort_values(['_score','action_id','dimension','indicator','city_indicator'],
                                      ascending=[False,True,True,True,True]).drop(columns='_score').reset_index(drop=True)
per_cell_df.to_csv(f'{BASE}/ranked_actions_per_cell.csv', index=False)

print(f'Wrote ranked_actions.csv ({len(ranked)} actions)')
print(f'Wrote ranked_actions_per_cell.csv ({len(per_cell_df)} rows — one per (action, cell, city_indicator) tuple)')
ranked.head(20)

,action_id,action_name,primary_outcome,primary_channel,sr15_options,match_strength,n_ac_cells,n_dims_scored,econ,tech,inst,soc,env,geo,score
0,icare_0012,Expand solar energy generation on municipal fa...,emissions_fuel_switch,solar_pv,Solar PV,direct,14,6,0.67,1.0,1.0,1.0,1.0,1.0,0.944
1,ipcc_0038,Stimulate solar energy production in industria...,emissions_fuel_switch,solar_pv,Solar PV,direct,14,6,0.67,1.0,1.0,1.0,1.0,1.0,0.944
2,icare_0050,Accelerate the adoption of Agroforestry',carbon_sequestration,agricultural_soil,Soil carbon sequestration & biochar,direct,7,5,0.67,1.0,NaN,1.0,1.0,1.0,0.933
3,icare_0045,"Promote organic, regenerative and agroecologic...",carbon_sequestration,agricultural_soil,Soil carbon sequestration & biochar,direct,7,5,0.67,1.0,NaN,1.0,1.0,1.0,0.933
4,ipcc_0062,Create partnerships to implement biochar plants.,carbon_sequestration,biochar,Soil carbon sequestration & biochar,direct,7,5,0.67,1.0,NaN,1.0,1.0,1.0,0.933
5,ipcc_0056,Reduce and prevent degradation and conversion ...,carbon_sequestration,agricultural_soil,Soil carbon sequestration & biochar,direct,7,5,0.67,1.0,NaN,1.0,1.0,1.0,0.933
6,ipcc_0041,Support research into carbon-neutral manufactu...,emissions_efficiency,industrial_process,Energy efficiency,direct,8,6,0.50,1.0,1.0,1.0,1.0,1.0,0.917
7,icare_0072,Improve Heat Recovery and Reuse through Heat E...,emissions_efficiency,industrial_process,Energy efficiency,direct,8,6,0.50,1.0,1.0,1.0,1.0,1.0,0.917
8,icare_0132,Encourage Local Agroecological Production and ...,emissions_fuel_switch,agricultural_soil,Sustainable intensification,partial,12,6,0.50,1.0,1.0,1.0,1.0,1.0,0.917
9,icare_0009,Deploy cogeneration systems in industries,emissions_efficiency,industrial_process,Energy efficiency,direct,8,6,0.50,1.0,1.0,1.0,1.0,1.0,0.917


## Final renamed output (`scoring_full_view.csv`)

Builds the audit-grade wide table directly from the canonical inputs (does not
read `scoring_full_view.csv`). Output columns match the team's house naming:
`global_*` for IPCC-side fields, `city_*` for city-side, and 0–1 scores at the
end (`city_adjusted_score`, `dimension_score`, `action_score`).

In [7]:
# Option-level Evidence + Agreement labels from SR1.5 SM tables
OPTION_EVIDENCE = {
    'Wind (on-shore & off-shore)':('Robust','Medium'),'Solar PV':('Robust','High'),'Bioenergy':('Robust','Medium'),
    'Electricity storage':('Robust','Medium'),'Power sector CCS':('Robust','High'),'Nuclear energy':('Robust','High'),
    'Smart grids':('Medium','Medium'),'BECCS':('Robust','Medium'),'DACCS':('Medium','Medium'),
    'Reduced food wastage':('Medium','High'),'Dietary shifts':('Medium','High'),
    'Sustainable intensification':('Medium','High'),'Afforestation & reforestation':('Robust','High'),
    'Soil carbon sequestration & biochar':('Robust','High'),'Enhanced weathering':('Medium','Low'),
    'Land-use & urban planning':('Robust','Medium'),
    'Electric cars and buses':('Medium','High'),'Sharing schemes':('Limited','Medium'),
    'Public transport':('Robust','Medium'),'Non-motorised transport':('Robust','High'),'Aviation & shipping':('Medium','Medium'),
    'Efficient appliances':('Medium','High'),'Low/zero-energy buildings':('Medium','High'),
    'Energy efficiency':('Robust','High'),'Bio-based & circularity':('Medium','Medium'),
    'Electrification & hydrogen':('Medium','High'),'Industrial CCS':('Robust','High'),
}
CODE_TO_RELATION = {
    'A':'barrier', 'B':'neutral', 'C':'supportive',
    'NA':'not applicable', 'NE':'no evidence', 'LE':'limited evidence',
}
CODE_TO_NUMERIC = {'A':1, 'B':2, 'C':3}
CODE_TO_SIGNAL  = {'A':-1, 'C':1}

LOCODE = 'CL CNE'
city_cat = {r['attribute_type']: r['attribute_category'] for _,r in city_df[city_df['locode']==LOCODE].iterrows()}
city_q   = {r['attribute_type']: QUINT[r['attribute_category']] for _,r in city_df[city_df['locode']==LOCODE].iterrows()}

real_bridges = ind_bridge[ind_bridge['city_indicator']!='(option-level constant)']
sm_table_by_opt = per_cell.groupby('option')['table'].first().to_dict()

rows = []
for _, a in actions.iterrows():
    aid = a['action_id']
    options = [o for o in a['sr15_options'].split('|') if not o.startswith('(')]
    if not options:
        continue
    for opt in options:
        family = OPTION_FAMILY.get(opt, '?')
        ev, ag = OPTION_EVIDENCE.get(opt, ('', ''))
        sm_table = sm_table_by_opt.get(opt, '')
        sm_section = SECTION.get(sm_table, '')
        ac_cells = per_cell[(per_cell['option']==opt) & (per_cell['code'].isin(['A','C']))]
        for _, c in ac_cells.iterrows():
            ind, code = c['indicator'], c['code']
            apps = real_bridges[(real_bridges['sr15_indicator']==ind) & (real_bridges['scope'].isin(['all', family]))]
            # Common cell-level fields
            base = {
                'action_id': aid,
                'action_name': a['action_name'],
                'sr15_option': opt,
                'match_strength': a['match_strength'],
                'option_evidence': ev,
                'option_agreement': ag,
                'dimension': c['dimension'],
                'global_indicator': ind,
                'global_relation': CODE_TO_RELATION.get(code, code),
                'sign': 0,
                'city_locode': LOCODE,
                'city_quintile_category': None,
                'city_capacity': None,
                'global_indicator_code_value': CODE_TO_NUMERIC.get(code),
                'global_indicator_signal':    CODE_TO_SIGNAL.get(code),
                'city_indicator_signed_score': None,
                'city_mean_signed_score':     None,
                'city_adjusted_score':        None,
                'dimension_score':            None,
                'action_score':               None,
                'city_indicator': None,
            }
            if apps.empty:
                rows.append(base)
                continue
            for _, b in apps.iterrows():
                ci = b['city_indicator']
                if ci in city_q:
                    cap = city_q[ci]; cat = city_cat[ci]
                    contrib = round(int(b['sign']) * (2*cap - 1), 3)
                else:
                    cap, cat, contrib = None, None, None
                rows.append({**base,
                    'city_indicator': ci,
                    'sign': int(b['sign']),
                    'city_quintile_category': cat,
                    'city_capacity': cap,
                    'city_indicator_signed_score': contrib,
                })

wide = pd.DataFrame(rows)

# Compute aggregates: city_mean_signed_score, city_adjusted_score, dimension_score, action_score
key_cell = ['action_id','sr15_option','dimension','global_indicator']
# Bridge mean per cell (across rows with a real sign)
valid = wide['sign'].isin([1,-1])
wide['city_mean_signed_score'] = wide[valid].groupby(key_cell)['city_indicator_signed_score'].transform('mean')
wide['city_mean_signed_score'] = wide.groupby(key_cell)['city_mean_signed_score'].transform('max')
# city_adjusted_score = ((signal + mean)/2 + 1)/2 if mean exists, else (signal+1)/2
def cell_score(row):
    s, m = row['global_indicator_signal'], row['city_mean_signed_score']
    if pd.notna(m):
        return round(((s + m)/2.0 + 1)/2.0, 3)
    return round((s + 1)/2.0, 3)
wide['city_adjusted_score'] = wide.apply(cell_score, axis=1)
# dimension_score = mean of distinct cell scores per (action, dimension)
cell_dedup = wide.drop_duplicates(subset=key_cell)[['action_id','dimension','city_adjusted_score']]
dim_scores_df = cell_dedup.groupby(['action_id','dimension'])['city_adjusted_score'].mean().reset_index().rename(columns={'city_adjusted_score':'dimension_score'})
wide = wide.drop(columns=['dimension_score']).merge(dim_scores_df, on=['action_id','dimension'], how='left')
wide['dimension_score'] = wide['dimension_score'].round(3)
# action_score = mean of dimension_score per action
act_scores = dim_scores_df.groupby('action_id')['dimension_score'].mean().reset_index().rename(columns={'dimension_score':'action_score'})
wide = wide.drop(columns=['action_score']).merge(act_scores, on='action_id', how='left')
wide['action_score'] = wide['action_score'].round(3)

# Final column order
wide = wide[[
    'action_id','action_name','sr15_option','match_strength',
    'option_evidence','option_agreement',
    'dimension','global_indicator','global_relation',
    'city_indicator','sign','city_locode','city_quintile_category','city_capacity',
    'global_indicator_code_value','global_indicator_signal',
    'city_indicator_signed_score','city_mean_signed_score',
    'city_adjusted_score','dimension_score','action_score',
]].sort_values(['action_score','action_id','dimension','global_indicator','city_indicator'],
               ascending=[False,True,True,True,True]).reset_index(drop=True)

wide.to_csv(f'{BASE}/feasibility_full_view.csv', index=False)
print(f'Wrote feasibility_full_view.csv ({len(wide)} rows × {len(wide.columns)} columns)')
wide.head(15)

Wrote feasibility_full_view.csv (1176 rows × 21 columns)


,action_id,action_name,sr15_option,match_strength,option_evidence,option_agreement,dimension,global_indicator,global_relation,city_indicator,...,city_locode,city_quintile_category,city_capacity,global_indicator_code_value,global_indicator_signal,city_indicator_signed_score,city_mean_signed_score,city_adjusted_score,dimension_score,action_score
0,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,economic,cost-effectiveness,supportive,median_household_income,...,CL CNE,very low,0.0,3,1,-1.0,-1.0,0.5,0.667,0.944
1,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,economic,cost-effectiveness,supportive,poverty_rate,...,CL CNE,very high,1.0,3,1,-1.0,-1.0,0.5,0.667,0.944
2,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,economic,distributional_effects,supportive,poverty_rate,...,CL CNE,very high,1.0,3,1,-1.0,-1.0,0.5,0.667,0.944
3,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,economic,employment_productivity,supportive,unemployment_rate,...,CL CNE,very high,1.0,3,1,1.0,1.0,1.0,0.667,0.944
4,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,environmental,air_pollution,supportive,NaN,...,CL CNE,NaN,NaN,3,1,NaN,NaN,1.0,1.000,0.944
5,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,environmental,biodiversity,supportive,NaN,...,CL CNE,NaN,NaN,3,1,NaN,NaN,1.0,1.000,0.944
6,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,environmental,water_use,supportive,NaN,...,CL CNE,NaN,NaN,3,1,NaN,NaN,1.0,1.000,0.944
7,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,geophysical,physical_feasibility,supportive,NaN,...,CL CNE,NaN,NaN,3,1,NaN,NaN,1.0,1.000,0.944
8,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,institutional,transparency_accountability,supportive,NaN,...,CL CNE,NaN,NaN,3,1,NaN,NaN,1.0,1.000,0.944
9,icare_0012,Expand solar energy generation on municipal fa...,Solar PV,direct,Robust,High,socio_cultural,intergenerational_equity,supportive,NaN,...,CL CNE,NaN,NaN,3,1,NaN,NaN,1.0,1.000,0.944
